In [ ]:
!pip install torch_geometric

In [11]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import scipy.io
from FVE_GCN_utils import load_surface_mesh
from matplotlib.tri import Triangulation


# LR


In [18]:
wd = Path("/niddk-data-central/mae_hr/FVE")
SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")

output_dir = wd / "LR_output"
B=50
top_percent = 0.1
def load_coefficients_from_csv(model_type, output_dir, B):
    filepath = output_dir / f'coefficients_{model_type}_boot{B}.csv'
    
    df = pd.read_csv(filepath)
    
    coef_dict = {}
    for b in range(1, B + 1):
        col_name = f'b{b}'
        if col_name in df.columns:
            coef_dict[b] = df[col_name].values
    
    return coef_dict


def get_top_features(coefficients, k=0.10):
    abs_coefs = np.abs(coefficients)
    n_features = len(abs_coefs)
    n_top = int(np.ceil(n_features * k))
    top_indices = np.argsort(abs_coefs)[::-1][:n_top]
    
    return top_indices


def extract_feature_importance(output_dir, B, model_configs):
    feature_counts = {}
    
    for model_name, config in model_configs.items():
        n_features = config['n_features']
        feature_counts[model_name] = np.zeros(n_features)
    
        coef_dict = load_coefficients_from_csv(model_name, output_dir, B)
        
        
        for b in range(1, B + 1):
            if b in coef_dict:
                coefficients = coef_dict[b]
                
                # Get top features (includes age/sex for regular models)
                top_indices = get_top_features(coefficients, k=k)
                
                # Increment count for these features
                feature_counts[model_name][top_indices] += 1
                
        n_selected = np.sum(feature_counts[model_name] > 0)
        print(f"  Total features selected at least once: {n_selected}")
        print(f"  Max selection count: {int(feature_counts[model_name].max())}")
    
    return feature_counts


def create_feature_dataframes(feature_counts, model_configs):
    feature_dfs = {}
    
    for model_name, counts in feature_counts.items():
        n_features = len(counts)
        config = model_configs[model_name]
        
        # Create base dataframe
        df = pd.DataFrame({
            'feature_id': np.arange(n_features),
            'count': counts,
            'proportion': counts / B,
            'model': model_name
        })
        
        # Add feature type and hemisphere labels
        if config['type'] == 'regular':
            # 10242 left + 10242 right + age + sex
            feature_types = ['vertex'] * 20484 + ['age', 'sex']
            hemispheres = ['left'] * 10242 + ['right'] * 10242 + ['NA', 'NA']
            vertex_nums = list(range(10242)) * 2 + [-1, -1]
        else:
            # 10242 left + 10242 right vertices only
            feature_types = ['vertex'] * 20484
            hemispheres = ['left'] * 10242 + ['right'] * 10242
            vertex_nums = list(range(10242)) * 2
        
        df['feature_type'] = feature_types
        df['hemisphere'] = hemispheres
        df['vertex_num'] = vertex_nums
        
        feature_dfs[model_name] = df
    
    return feature_dfs


def save_results(feature_counts, feature_dfs, output_dir, B):
    np.save(output_dir / f'vis_output/feature_counts_boot{B}.npy', feature_counts)
    
    # Save as pickle
    with open(output_dir / f'vis_output/feature_importance_boot{B}.pkl', 'wb') as f:
        pickle.dump({
            'feature_counts': feature_counts,
            'feature_dfs': feature_dfs,
            'B': B,
            'k': top_percent
        }, f)


def create_summary_statistics(feature_dfs, B):
    summary_data = []
    
    for model_name, df in feature_dfs.items():
        brain_features = df[df['feature_type'] == 'vertex']
        
        summary_data.append({
            'Model': model_name,
            'n_total_Vertices_Selected': int(np.sum(brain_features['count'] > 0)),
            'n_vertices>50pct': int(np.sum(brain_features['count'] >= B * 0.5)),
            'n_vertices>75pct': int(np.sum(brain_features['count'] >= B * 0.75)),
            'n_vertices=50times': int(np.sum(brain_features['count'] == B)),
            'max_selection_count': int(brain_features['count'].max())
        })
        
        # For regular models, add covariate info
        if model_name in ['LASSO', 'Ridge']:
            age_count = df[df['feature_type'] == 'age']['count'].values[0]
            sex_count = df[df['feature_type'] == 'sex']['count'].values[0]
            summary_data[-1]['age_selected_count'] = int(age_count)
            summary_data[-1]['sex_selected_count'] = int(sex_count)
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(output_dir / f'vis_output/feature_importance_summary_boot{B}.csv', index=False)
    
    print(summary_df.to_string(index=False))

    
    return summary_df



def plot_selection_histograms(feature_dfs, output_dir, B):
    """Create histograms of selection frequencies"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (model_name, df) in enumerate(feature_dfs.items()):
        ax = axes[idx]
        
        # Get brain vertices that were selected at least once
        brain_data = df[df['feature_type'] == 'vertex']
        selected_counts = brain_data[brain_data['count'] > 0]['count'].values
        
        if len(selected_counts) > 0:
            ax.hist(selected_counts, bins=min(50, B), color='steelblue', 
                   alpha=0.7, edgecolor='black')
            ax.axvline(B * 0.5, color='red', linestyle='--', linewidth=2, 
                      label=f'50% ({B*0.5:.0f})')
            ax.axvline(B * 0.75, color='darkred', linestyle='--', linewidth=2, 
                      label=f'75% ({B*0.75:.0f})')
        
        ax.set_xlabel('Number of Times Selected', fontsize=10)
        ax.set_ylabel('Number of Vertices', fontsize=10)
        ax.set_title(f'{model_name}\n{len(selected_counts)} vertices selected', 
                    fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / f'LR_selection_histograms_boot{B}.png', 
               dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Histogram plot saved: selection_histograms_boot{B}.png")


def plot_model_comparison(summary_df, output_dir, B):
    fig, ax = plt.subplots(figsize=(14, 6))
    
    models = summary_df['Model'].values
    x = np.arange(len(models))
    width = 0.2
    
    ax.bar(x - 1.5*width, summary_df['n_total_Vertices_Selected'], width, 
           label='Any selection', alpha=0.8, color='lightblue')
    ax.bar(x - 0.5*width, summary_df['n_vertices>50pct'], width, 
           label='>50% iterations', alpha=0.8, color='orange')
    ax.bar(x + 0.5*width, summary_df['n_vertices>75pct'], width, 
           label='>75% iterations', alpha=0.8, color='red')
    ax.bar(x + 1.5*width, summary_df['n_vertices=50times'], width, 
           label='All iterations', alpha=0.8, color='darkred')
    
    ax.set_xlabel('Model Type', fontsize=12)
    ax.set_ylabel('Number of Vertices', fontsize=12)
    ax.set_title(f'Feature Consistency Across Models (B={B})', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=0, ha='right')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / f'LR_model_comparison_boot{B}.png', 
               dpi=300, bbox_inches='tight')
    plt.close()
    


def plot_brain_heatmap(feature_dfs, model_types, B, output_dir):
    
    fig, axes = plt.subplots(6, 1, figsize=(16, 24))
    
    # Create white to red colormap
    cmap = LinearSegmentedColormap.from_list('white_red', ['white', 'red'])
    
    # Display names for models
    model_display_names = {
        'LASSO': 'LASSO',
        'LASSO_partial': 'LASSO partial',
        'LASSO_partial_tsa': 'LASSO TSA partial',
        'Ridge': 'Ridge',
        'Ridge_partial': 'Ridge partial',
        'Ridge_partial_tsa': 'Ridge TSA partial'
    }
    
    for idx, model_type in enumerate(model_types):
        if model_type not in feature_dfs:
            continue
        
        print(f"\nProcessing {model_type}...")
        
        # Extract hemisphere data
        df = feature_dfs[model_type]
        brain_features = df[df['feature_type'] == 'vertex']
        
        lh_data = brain_features[brain_features['hemisphere'] == 'left'].copy()
        rh_data = brain_features[brain_features['hemisphere'] == 'right'].copy()
        
        lh_counts = np.zeros(10242)
        rh_counts = np.zeros(10242)
        
        for _, row in lh_data.iterrows():
            lh_counts[int(row['vertex_num'])] = row['count']
        
        for _, row in rh_data.iterrows():
            rh_counts[int(row['vertex_num'])] = row['count']
        
        # Print statistics

        # Plot in subplot
        ax = axes[idx]
        
        # Stack left and right hemispheres horizontally
        combined_counts = np.hstack([lh_counts.reshape(1, -1), rh_counts.reshape(1, -1)])
        
        im = ax.imshow(combined_counts, aspect='auto', cmap=cmap, vmin=0, vmax=B)
        
        # Add vertical line to separate hemispheres
        ax.axvline(x=10242 - 0.5, color='black', linewidth=2, linestyle='--')
        
        # Labels - use display name
        display_name = model_display_names.get(model_type, model_type)
        ax.set_ylabel(f'{display_name}', fontsize=14, fontweight='bold')
        
        # Only show x-axis label on bottom plot
        if idx == len(model_types) - 1:
            ax.set_xlabel('Vertex Index', fontsize=11)
            # Add hemisphere labels only on bottom
            ax.text(10242/2, -0.15, 'Left\nHemisphere', 
                    transform=ax.get_xaxis_transform(), fontsize=12, 
                    ha='center', va='center')
            ax.text(10242 + 10242/2, -0.15, 'Right\nHemisphere', 
                    transform=ax.get_xaxis_transform(), fontsize=12, 
                    ha='center', va='center')
        else:
            ax.set_xticklabels([])
    
    # Add single colorbar for all subplots
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax, label=f'Selection Count (out of {B})')
    
    plt.suptitle('LASSO and Ridge top 10% vertices selection', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.savefig(output_dir / f'LR_brain_heatmap_boot{B}.png', dpi=450, bbox_inches='tight')
    plt.close()



In [20]:
B = 50  
k = 0.10 

# Feature dimensions
N_VERTICES_PER_HEMI = 10242  # vertices per hemisphere (0-10241)
N_VERTICES = N_VERTICES_PER_HEMI * 2  # 20484 total
N_COVARIATES = 2  # age and sex
N_FEATURES_REGULAR = N_VERTICES + N_COVARIATES  # 20486
N_FEATURES_PARTIAL = N_VERTICES  # 20484

model_types = [
    'LASSO', 'LASSO_partial', 'LASSO_partial_tsa',
    'Ridge', 'Ridge_partial', 'Ridge_partial_tsa'
]

# Model configurations
MODEL_CONFIGS = {
    'LASSO': {'type': 'regular', 'n_features': N_FEATURES_REGULAR},
    'LASSO_partial': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'LASSO_partial_tsa': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'Ridge': {'type': 'regular', 'n_features': N_FEATURES_REGULAR},
    'Ridge_partial': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'Ridge_partial_tsa': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL}
}

coef_files = list(output_dir.glob('coefficients_*_boot_50.csv'))
feature_counts = extract_feature_importance(output_dir, B, MODEL_CONFIGS)
feature_dfs = create_feature_dataframes(feature_counts, MODEL_CONFIGS)
summary_df = create_summary_statistics(feature_dfs, B)
save_results(feature_counts, feature_dfs, output_dir, B)


#visualizations
plot_selection_histograms(feature_dfs, output_dir, B)
plot_model_comparison(summary_df, output_dir, B)
with open(output_dir / f'vis_output/feature_importance_boot{B}.pkl', 'rb') as f:
    data = pickle.load(f)
    feature_dfs = data['feature_dfs']
plot_brain_heatmap(feature_dfs, model_types, B, output_dir)


  Total features selected at least once: 12015
  Max selection count: 50
  Total features selected at least once: 12390
  Max selection count: 50
  Total features selected at least once: 15729
  Max selection count: 47
  Total features selected at least once: 9455
  Max selection count: 50
  Total features selected at least once: 9596
  Max selection count: 50
  Total features selected at least once: 10366
  Max selection count: 50
            Model  n_total_Vertices_Selected  n_vertices>50pct  n_vertices>75pct  n_vertices=50times  max_selection_count  age_selected_count  sex_selected_count
            LASSO                      12013              1384              1067                 723                   50                50.0                50.0
    LASSO_partial                      12390              1357              1069                 642                   50                 NaN                 NaN
LASSO_partial_tsa                      15729               835               1

/tmp/ipykernel_11233/3954862374.py:280: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
